In [ ]:
import os, glob, zipfile, warnings, random
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageFile
from scipy import ndimage
from scipy.stats import skew, kurtosis
ImageFile.LOAD_TRUNCATED_IMAGES = True
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110
SEED = 42; random.seed(SEED); np.random.seed(SEED)

# EDA - Klasifikasi Citra Aksara Tradisional Nusantara (INFEST XII 2026)

Dataset ini **tidak** berbentuk folder-per-kelas seperti kebanyakan dataset citra:
labelnya ada di `train.csv` (`image_id,label`) sementara gambarnya tersebar di
beberapa folder. Sel berikut menemukan file dan memetakannya ke label.

**Catatan penting:** EDA ini memberi porsi besar pada perbandingan TRAIN vs TEST.
Pada lomba ini jarak OOF-leaderboard mencapai 0.13-0.16, dan penyebabnya ada di
perbedaan kedua himpunan itu - bukan di kesulitan klasifikasinya.

In [ ]:
# ---- temukan CSV dan seluruh file gambar (tahan thd struktur folder apa pun)
SEARCH = [p for p in ["/kaggle/input", "/kaggle/working", ".", "./data", "/content"]
          if os.path.isdir(p)]

def find_csv(names):
    for root in SEARCH:
        for dp, _, fs in os.walk(root, followlinks=True):
            for f in fs:
                if f.lower() in names: return os.path.join(dp, f)
    return None

train_csv = find_csv({"train.csv"}); test_csv = find_csv({"test.csv"})
tr_df = pd.read_csv(train_csv); te_df = pd.read_csv(test_csv)

# ekstrak zip kalau gambarnya masih terkompresi
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
for root in list(SEARCH):
    for z in glob.glob(os.path.join(root, "*.zip")):
        out = os.path.join(WORK, "_imgs")
        if not os.path.isdir(out): os.makedirs(out, exist_ok=True)
        tag = os.path.splitext(os.path.basename(z))[0]
        if not os.path.isdir(os.path.join(out, tag)):
            with zipfile.ZipFile(z) as zf:
                zf.extractall(os.path.join(out, tag),
                              members=[m for m in zf.namelist() if not m.startswith("__MACOSX")])
if os.path.isdir(os.path.join(WORK, "_imgs")): SEARCH.append(os.path.join(WORK, "_imgs"))

EXT = (".png",".jpg",".jpeg",".webp",".bmp",".gif",".tif",".tiff")
IDX, STEM = {}, {}
for root in SEARCH:
    for dp, _, fs in os.walk(root, followlinks=True):
        for f in fs:
            if f.lower().endswith(EXT):
                full = os.path.join(dp, f)
                IDX.setdefault(f, []).append(full)
                STEM.setdefault(os.path.splitext(f)[0], []).append(full)

def resolve(name, split):
    b = os.path.basename(str(name))
    c = IDX.get(b) or STEM.get(os.path.splitext(b)[0])   # CSV kadang beda ekstensi
    if not c: return None
    if len(c) == 1: return c[0]
    pref = [p for p in c if f"{os.sep}{split}" in p.lower()]
    return (pref or c)[0]

tr_df["path"] = [resolve(x, "train") for x in tr_df.image_id]
te_df["path"] = [resolve(x, "test")  for x in te_df.image_id]
tr_df = tr_df[tr_df.path.notna()].reset_index(drop=True)
te_df = te_df[te_df.path.notna()].reset_index(drop=True)
CLASSES = sorted(tr_df.label.unique())
print(f"train {len(tr_df)} gambar | test {len(te_df)} gambar | {len(CLASSES)} kelas")
print(CLASSES)

In [ ]:
# ---- util bersama
def load_gray(p, size=None):
    """Baca 1 gambar sbg array grayscale float."""
    try:
        with Image.open(p) as im:
            im = im.convert("L")
            if size: im = im.resize(size, Image.BILINEAR)
            elif max(im.size) > 512: im.thumbnail((512, 512))
            return np.asarray(im, dtype=np.float64)
    except Exception:
        return None

def load_rgb(p, size=None):
    try:
        with Image.open(p) as im:
            im = im.convert("RGB")
            if size: im = im.resize(size, Image.BILINEAR)
            elif max(im.size) > 512: im.thumbnail((512, 512))
            return np.asarray(im, dtype=np.float64)
    except Exception:
        return None

def sample_paths(df, n, cls=None, seed=SEED):
    d = df[df.label == cls] if cls is not None else df
    return d.sample(min(n, len(d)), random_state=seed).path.tolist()

PAL = sns.color_palette("viridis", len(CLASSES))
CCOL = dict(zip(CLASSES, PAL))

# 1. DISTRIBUSI SETIAP KELAS

In [ ]:
vc = tr_df.label.value_counts().reindex(CLASSES)
dfk = pd.DataFrame({"Kelas": CLASSES, "Jumlah": vc.values})

plt.figure(figsize=(9,5))
ax = sns.barplot(data=dfk, x="Kelas", y="Jumlah", palette="viridis", edgecolor="black")
for i, v in enumerate(dfk.Jumlah):
    ax.text(i, v + 8, f"{v}\n({v/len(tr_df)*100:.1f}%)", ha="center", fontsize=9)
plt.title(f"Distribusi kelas TRAIN (n={len(tr_df)}) - metrik lomba: F1 Macro")
plt.ylim(0, dfk.Jumlah.max()*1.18); plt.tight_layout(); plt.show()

print(f"imbalance ratio mayoritas/minoritas = {vc.max()/vc.min():.2f}x")
print("Karena metriknya MACRO-F1, kelas terkecil (pegon) berbobot sama dengan")
print("kelas terbesar (jawi). Satu kesalahan di pegon 2.5x lebih mahal.")

# 2. SAMPEL PER KELAS

Perhatikan **bentuk** gambarnya, bukan hanya isinya: sebagian besar train adalah
potongan teks memanjang (strip) dengan latar putih bersih.

In [ ]:
N = 6
fig, axes = plt.subplots(len(CLASSES), N, figsize=(N*2.6, len(CLASSES)*1.5))
for i, cls in enumerate(CLASSES):
    for j, p in enumerate(sample_paths(tr_df, N, cls)):
        a = load_rgb(p)
        ax = axes[i, j]
        if a is not None: ax.imshow(a.astype(np.uint8))
        ax.axis("off")
        if j == 0: ax.set_title(cls, loc="left", fontsize=10, color="darkred")
plt.suptitle("Sampel TRAIN per kelas", y=1.002); plt.tight_layout(); plt.show()

# 3. ANALISIS WARNA (RGB)

Bagian ini yang paling penting di dataset ini. Bandingkan histogram TRAIN vs TEST,
bukan hanya antar kelas.

In [ ]:
def rgb_hist(paths, label, ax):
    hs = [np.zeros(256) for _ in range(3)]
    for p in paths:
        a = load_rgb(p)
        if a is None: continue
        for c in range(3):
            hs[c] += np.histogram(a[..., c], bins=256, range=(0,256))[0]
    for c, col in zip(range(3), ("red","green","blue")):
        ax.plot(hs[c]/max(1, hs[c].sum()), color=col, alpha=.75, lw=1.2)
    ax.set_title(label, fontsize=10); ax.set_yscale("log")

fig, axes = plt.subplots(1, 2, figsize=(12,3.5))
rgb_hist(sample_paths(tr_df, 250), f"TRAIN (n=250)", axes[0])
rgb_hist(sample_paths(te_df, 250), f"TEST (n=250)",  axes[1])
plt.suptitle("Histogram RGB: TRAIN vs TEST"); plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, len(CLASSES), figsize=(len(CLASSES)*2.4, 2.6))
for ax, cls in zip(axes, CLASSES):
    rgb_hist(sample_paths(tr_df, 60, cls), cls, ax)
plt.suptitle("Histogram RGB per kelas (TRAIN)"); plt.tight_layout(); plt.show()

# 4. GRAYSCALE vs BERWARNA - jarak domain terbesar

In [ ]:
def saturasi(paths):
    out = []
    for p in paths:
        a = load_rgb(p, size=(160,160))
        if a is None: continue
        out.append(float(np.abs(a - a.mean(2, keepdims=True)).mean()))
    return np.array(out)

s_tr, s_te = saturasi(sample_paths(tr_df, 400)), saturasi(sample_paths(te_df, 400))

fig, axes = plt.subplots(1, 2, figsize=(12,3.6))
axes[0].hist(s_tr, bins=40, alpha=.65, label="TRAIN", color="steelblue", density=True)
axes[0].hist(s_te, bins=40, alpha=.65, label="TEST",  color="darkorange", density=True)
axes[0].axvline(2, ls="--", c="k", lw=1); axes[0].set_yscale("log")
axes[0].set_xlabel("saturasi rata-rata (0 = abu-abu murni)"); axes[0].legend()
axes[0].set_title("Sebaran saturasi warna")

bars = pd.DataFrame({"himpunan":["TRAIN","TEST"],
                     "abu-abu %":[100*(s_tr<2).mean(), 100*(s_te<2).mean()]})
sns.barplot(data=bars, x="himpunan", y="abu-abu %", ax=axes[1],
            palette=["steelblue","darkorange"], edgecolor="black")
for i,v in enumerate(bars["abu-abu %"]): axes[1].text(i, v+1, f"{v:.1f}%", ha="center")
axes[1].set_ylim(0,110); axes[1].set_title("Porsi citra abu-abu murni")
plt.tight_layout(); plt.show()

print(f"TRAIN abu-abu {100*(s_tr<2).mean():.1f}%  vs  TEST abu-abu {100*(s_te<2).mean():.1f}%")
print("Model yang dilatih pada data hampir seluruhnya hitam-putih akan menghadapi")
print("test yang mayoritas berwarna. Solusi termurah: paksa SEMUA input ke abu-abu.")

# 5. CONTRAST & BRIGHTNESS

In [ ]:
def bc(paths):
    b, c = [], []
    for p in paths:
        a = load_gray(p)
        if a is None: continue
        b.append(a.mean()); c.append(a.std())
    return np.array(b), np.array(c)

rows = []
for cls in CLASSES:
    b, c = bc(sample_paths(tr_df, 120, cls))
    rows.append(dict(kelas=cls, brightness=b.mean(), contrast=c.mean(), n=len(b)))
b_te, c_te = bc(sample_paths(te_df, 300))
dfbc = pd.DataFrame(rows)

fig, axes = plt.subplots(1, 2, figsize=(12,3.6))
sns.barplot(data=dfbc, x="kelas", y="brightness", ax=axes[0], palette="viridis", edgecolor="black")
axes[0].axhline(b_te.mean(), ls="--", c="darkorange", lw=2, label=f"TEST ({b_te.mean():.0f})")
axes[0].legend(); axes[0].set_title("Brightness rata-rata per kelas (TRAIN)")
sns.barplot(data=dfbc, x="kelas", y="contrast", ax=axes[1], palette="viridis", edgecolor="black")
axes[1].axhline(c_te.mean(), ls="--", c="darkorange", lw=2, label=f"TEST ({c_te.mean():.0f})")
axes[1].legend(); axes[1].set_title("Contrast (std) rata-rata per kelas (TRAIN)")
for ax in axes: ax.tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.show()
display(dfbc.round(2))

# 6. BLUR DETECTION & LAPLACIAN VARIANCE

Nilai kecil = buram. Ini jarak domain terbesar kedua setelah warna.

In [ ]:
def lapvar(paths, size=(256,256)):
    out = []
    for p in paths:
        a = load_gray(p, size=size)
        if a is None: continue
        out.append(float(ndimage.laplace(a).var()))
    return np.array(out)

lv = {cls: lapvar(sample_paths(tr_df, 100, cls)) for cls in CLASSES}
lv_te = lapvar(sample_paths(te_df, 300))

fig, ax = plt.subplots(figsize=(11,4.5))
TICKS = CLASSES + ["TEST\n(semua)"]
bp = ax.boxplot([lv[c] for c in CLASSES] + [lv_te], patch_artist=True, showmeans=True)
ax.set_xticks(range(1, len(TICKS)+1)); ax.set_xticklabels(TICKS)   # 'labels=' dihapus
                                                                   # di matplotlib 3.9+
for i, box in enumerate(bp["boxes"]):
    box.set_facecolor("darkorange" if i == len(CLASSES) else "lightblue"); box.set_alpha(.6)
ax.set_yscale("log"); ax.set_ylabel("variansi Laplacian (log)")
ax.set_title("Ketajaman: tiap kelas TRAIN vs seluruh TEST")
plt.tight_layout(); plt.show()

m_tr = np.median(np.concatenate([lv[c] for c in CLASSES])); m_te = np.median(lv_te)
print(f"median TRAIN = {m_tr:,.0f} | median TEST = {m_te:,.0f} -> test {m_tr/m_te:.1f}x lebih buram")
print("Konsekuensi: augmentasi blur pada TRAIN perlu DITERA ke angka ini, bukan ditebak.")

# 7. PIXEL STATISTICS (mean, std, skew, kurtosis)

In [ ]:
recs = []
for cls in CLASSES:
    px = []
    for p in sample_paths(tr_df, 40, cls):
        a = load_gray(p, size=(96,96))
        if a is not None: px.append(a.ravel())
    if not px: continue
    arr = np.concatenate(px)
    recs.append(dict(kelas=cls, mean=arr.mean(), std=arr.std(),
                     skew=skew(arr), kurtosis=kurtosis(arr)))
px_te = np.concatenate([load_gray(p,(96,96)).ravel()
                        for p in sample_paths(te_df, 120) if load_gray(p,(96,96)) is not None])
recs.append(dict(kelas="** TEST **", mean=px_te.mean(), std=px_te.std(),
                 skew=skew(px_te), kurtosis=kurtosis(px_te)))
dfpx = pd.DataFrame(recs)
display(dfpx.round(3))

fig, axes = plt.subplots(1, 4, figsize=(15,3))
for ax, col in zip(axes, ["mean","std","skew","kurtosis"]):
    cols = ["darkorange" if k.startswith("**") else "steelblue" for k in dfpx.kelas]
    ax.bar(dfpx.kelas, dfpx[col], color=cols, edgecolor="black")
    ax.set_title(col); ax.tick_params(axis="x", rotation=90)
plt.suptitle("Statistik piksel (skew tinggi = didominasi latar terang)")
plt.tight_layout(); plt.show()

# 8. CITRA RATA-RATA & STANDAR DEVIASI PER KELAS

In [ ]:
RT = (128,128)
fig, axes = plt.subplots(2, len(CLASSES), figsize=(len(CLASSES)*2.1, 4.6))
MEANIMG = {}
for i, cls in enumerate(CLASSES):
    st = [load_gray(p, RT) for p in sample_paths(tr_df, 150, cls)]
    st = np.stack([x for x in st if x is not None])
    MEANIMG[cls] = st.mean(0)
    axes[0,i].imshow(st.mean(0), cmap="gray"); axes[0,i].set_title(cls, fontsize=9)
    axes[1,i].imshow(st.std(0),  cmap="magma")
    axes[0,i].axis("off"); axes[1,i].axis("off")
axes[0,0].set_ylabel("MEAN"); axes[1,0].set_ylabel("STD")
plt.suptitle("Baris atas: citra rata-rata | Baris bawah: standar deviasi")
plt.tight_layout(); plt.show()

# 9. PAIRWISE CLASS SIMILARITY

Pasangan dgn kemiripan tertinggi = pasangan yang paling sering tertukar.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
V = np.stack([MEANIMG[c].ravel() for c in CLASSES])
S = cosine_similarity(V)
plt.figure(figsize=(6.5,5.2))
sns.heatmap(S, annot=True, fmt=".3f", xticklabels=CLASSES, yticklabels=CLASSES,
            cmap="rocket_r", vmin=S[S<1].min(), vmax=S[S<1].max())
plt.title("Cosine similarity antar citra rata-rata kelas"); plt.tight_layout(); plt.show()

iu = np.triu_indices(len(CLASSES), 1)
pairs = sorted(zip(S[iu], [(CLASSES[a],CLASSES[b]) for a,b in zip(*iu)]), reverse=True)
print("5 pasangan paling mirip:")
for v,(a,b) in pairs[:5]: print(f"  {a:9s} <-> {b:9s} {v:.4f}")

# 10. SELF-SIMILARITY (rotasi & flip)

Penting untuk memutuskan augmentasi. Aksara punya ORIENTASI bermakna:
jawi/pegon ditulis kanan-ke-kiri. Kalau flip menghasilkan kemiripan rendah,
berarti flip MERUSAK informasi kelas dan tidak boleh dipakai.

In [ ]:
def selfsim(cls, n=40):
    out = {"rot90":[], "rot180":[], "flip_h":[], "flip_v":[]}
    for p in sample_paths(tr_df, n, cls):
        a = load_gray(p, (64,64))
        if a is None: continue
        a = a/255.0
        for k, t in (("rot90",np.rot90(a,1)), ("rot180",np.rot90(a,2)),
                     ("flip_h",np.fliplr(a)), ("flip_v",np.flipud(a))):
            if t.shape != a.shape: continue
            out[k].append(float(np.corrcoef(a.ravel(), t.ravel())[0,1]))
    return {k: float(np.mean(v)) if v else np.nan for k,v in out.items()}

dfss = pd.DataFrame([dict(kelas=c, **selfsim(c)) for c in CLASSES]).set_index("kelas")
plt.figure(figsize=(7,4.2))
sns.heatmap(dfss, annot=True, fmt=".3f", cmap="coolwarm", center=0)
plt.title("Korelasi citra asli vs hasil transformasi"); plt.tight_layout(); plt.show()
print("Nilai rendah = transformasi MENGUBAH citra secara berarti -> JANGAN dipakai")
print("sebagai augmentasi. Di dataset ini flip horizontal khususnya berbahaya")
print("karena jawi & pegon adalah aksara turunan Arab yang dibaca kanan-ke-kiri.")

# 11. GLCM TEXTURE FEATURES

In [ ]:
from skimage.feature import graycomatrix, graycoprops
PROPS = ["contrast","homogeneity","energy","correlation"]

def glcm_rows(paths, tag, n=40):
    out = []
    for p in paths[:n]:
        a = load_gray(p, (128,128))
        if a is None: continue
        g = graycomatrix(a.astype(np.uint8), distances=[1], angles=[0],
                         levels=256, symmetric=True, normed=True)
        out.append(dict(kelas=tag, **{k: float(graycoprops(g,k)[0,0]) for k in PROPS}))
    return out

rows = []
for cls in CLASSES: rows += glcm_rows(sample_paths(tr_df, 40, cls), cls)
rows += glcm_rows(sample_paths(te_df, 80), "** TEST **", n=80)
dfg = pd.DataFrame(rows)

fig, axes = plt.subplots(1, 4, figsize=(16,3.4))
for ax, k in zip(axes, PROPS):
    sns.boxplot(data=dfg, x="kelas", y=k, ax=ax, palette="viridis")
    ax.tick_params(axis="x", rotation=90); ax.set_title(k)
plt.suptitle("GLCM texture - TEST disandingkan sbg pembanding")
plt.tight_layout(); plt.show()
display(dfg.groupby("kelas")[PROPS].mean().round(4))

# 12. EDGE DENSITY / GRADIENT MAGNITUDE

In [ ]:
def gradmag(paths, n=80):
    out = []
    for p in paths[:n]:
        a = load_gray(p, (128,128))
        if a is None: continue
        gx, gy = ndimage.sobel(a, 0), ndimage.sobel(a, 1)
        out.append(float(np.sqrt(gx**2+gy**2).mean()))
    return np.array(out)

gm = {c: gradmag(sample_paths(tr_df, 80, c)) for c in CLASSES}
gm_te = gradmag(sample_paths(te_df, 200), n=200)
dfgm = pd.DataFrame([dict(kelas=c, grad=v) for c in CLASSES for v in gm[c]] +
                    [dict(kelas="** TEST **", grad=v) for v in gm_te])
plt.figure(figsize=(10,4))
sns.violinplot(data=dfgm, x="kelas", y="grad", palette="viridis", cut=0)
plt.xticks(rotation=30); plt.title("Kerapatan tepi (Sobel) - proksi ketebalan & kejelasan coretan")
plt.tight_layout(); plt.show()

# 13. FFT / ANALISIS FREKUENSI

In [ ]:
def fft_energy(paths, n=40):
    lo, hi = [], []
    for p in paths[:n]:
        a = load_gray(p, (128,128))
        if a is None: continue
        m = np.log1p(np.abs(np.fft.fftshift(np.fft.fft2(a))))
        cy, cx = 64, 64
        Y, X = np.ogrid[:128,:128]
        r = np.sqrt((Y-cy)**2 + (X-cx)**2)
        lo.append(m[r<=16].mean()); hi.append(m[r>40].mean())
    return np.array(lo), np.array(hi)

rows = []
for cls in CLASSES:
    l,h = fft_energy(sample_paths(tr_df, 40, cls)); rows.append(dict(kelas=cls, rendah=l.mean(), tinggi=h.mean()))
l,h = fft_energy(sample_paths(te_df, 80), n=80); rows.append(dict(kelas="** TEST **", rendah=l.mean(), tinggi=h.mean()))
dff = pd.DataFrame(rows).set_index("kelas")
dff.plot(kind="bar", figsize=(9,3.6), edgecolor="black")
plt.ylabel("energi log-magnitude"); plt.title("Energi frekuensi RENDAH vs TINGGI")
plt.tight_layout(); plt.show()
print("Energi frekuensi tinggi rendah pada TEST = detail halus hilang (buram).")

# 14. SPEKTRUM FOURIER RATA-RATA

In [ ]:
def avg_spec(paths, n=30):
    acc = None
    for p in paths[:n]:
        a = load_gray(p, (128,128))
        if a is None: continue
        m = np.log1p(np.abs(np.fft.fftshift(np.fft.fft2(a))))
        acc = m if acc is None else acc + m
    return acc/max(1,n)

fig, axes = plt.subplots(1, len(CLASSES)+1, figsize=((len(CLASSES)+1)*2.1, 2.4))
for ax, cls in zip(axes, CLASSES):
    ax.imshow(avg_spec(sample_paths(tr_df, 30, cls)), cmap="inferno")
    ax.set_title(cls, fontsize=9); ax.axis("off")
axes[-1].imshow(avg_spec(sample_paths(te_df, 60), n=60), cmap="inferno")
axes[-1].set_title("TEST", fontsize=9, color="darkorange"); axes[-1].axis("off")
plt.suptitle("Spektrum Fourier rata-rata"); plt.tight_layout(); plt.show()

# 15. CLASS SEPARABILITY (PCA)

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
FS = (48,48)
X, Y = [], []
for cls in CLASSES:
    for p in sample_paths(tr_df, 90, cls):
        a = load_gray(p, FS)
        if a is not None: X.append(a.ravel()/255.0); Y.append(cls)
Xte = [load_gray(p, FS).ravel()/255.0 for p in sample_paths(te_df, 250)
       if load_gray(p, FS) is not None]
X = np.array(X); Xte = np.array(Xte)
sc = StandardScaler().fit(X)
pc = PCA(n_components=2, random_state=SEED).fit(sc.transform(X))
Z, Zte = pc.transform(sc.transform(X)), pc.transform(sc.transform(Xte))

fig, axes = plt.subplots(1, 2, figsize=(13,5))
for cls in CLASSES:
    m = np.array(Y) == cls
    axes[0].scatter(Z[m,0], Z[m,1], s=9, alpha=.6, label=cls, color=CCOL[cls])
axes[0].legend(fontsize=8, markerscale=1.6); axes[0].set_title("PCA - TRAIN per kelas")
axes[1].scatter(Z[:,0], Z[:,1], s=9, alpha=.35, color="steelblue", label="TRAIN")
axes[1].scatter(Zte[:,0], Zte[:,1], s=9, alpha=.55, color="darkorange", label="TEST")
axes[1].legend(); axes[1].set_title("PCA - TRAIN vs TEST (perhatikan pergeserannya)")
for ax in axes: ax.set_xlabel(f"PC1 ({pc.explained_variance_ratio_[0]*100:.1f}%)"); ax.set_ylabel(f"PC2 ({pc.explained_variance_ratio_[1]*100:.1f}%)")
plt.tight_layout(); plt.show()

# 16. DUA POPULASI: STRIP vs BLOK (aspect ratio)

Komposisi kedua populasi ini BERBEDA antara train dan test. Akibatnya angka OOF
polos terlalu optimistis - harus ditimbang ke komposisi test.

In [ ]:
def ars(df, n=1500):
    out = []
    for p in df.sample(min(n,len(df)), random_state=SEED).path:
        try:
            with Image.open(p) as im: out.append(im.size[0]/max(1,im.size[1]))
        except Exception: pass
    return np.array(out)

ar_tr, ar_te = ars(tr_df), ars(te_df)
fig, axes = plt.subplots(1, 2, figsize=(12,3.8))
axes[0].hist(np.log10(ar_tr), bins=50, alpha=.6, label="TRAIN", density=True, color="steelblue")
axes[0].hist(np.log10(ar_te), bins=50, alpha=.6, label="TEST",  density=True, color="darkorange")
axes[0].axvline(np.log10(3), ls="--", c="k"); axes[0].legend()
axes[0].set_xlabel("log10(aspect ratio)"); axes[0].set_title("Sebaran aspect ratio")

comp = pd.DataFrame({"himpunan":["TRAIN","TEST"],
                     "blok %":[100*(ar_tr<3).mean(), 100*(ar_te<3).mean()]})
sns.barplot(data=comp, x="himpunan", y="blok %", ax=axes[1],
            palette=["steelblue","darkorange"], edgecolor="black")
for i,v in enumerate(comp["blok %"]): axes[1].text(i, v+1, f"{v:.1f}%", ha="center")
axes[1].set_title("Porsi populasi 'blok' (AR<3)"); axes[1].set_ylim(0,100)
plt.tight_layout(); plt.show()

arc = pd.DataFrame([dict(kelas=c, AR_median=np.median(ars(tr_df[tr_df.label==c], 400)),
                         blok_pct=100*(ars(tr_df[tr_df.label==c],400)<3).mean()) for c in CLASSES])
display(arc.round(2))

# 17. DETEKSI OKLUSI BUATAN

Sebagian citra TEST diberi tambalan poligon abu-abu yang menutupi teks.
Ini augmentasi buatan yang TIDAK ADA di train.

In [ ]:
def occl_frac(paths, n=300):
    out = []
    for p in paths[:n]:
        a = load_gray(p, (256,256))
        if a is None: continue
        m  = ndimage.uniform_filter(a,5); m2 = ndimage.uniform_filter(a*a,5)
        sd = np.sqrt(np.maximum(m2-m*m,0))
        mid = (sd<1.5)&(a>60)&(a<200)
        lab,k = ndimage.label(mid)
        out.append(0.0 if k==0 else float(ndimage.sum(mid,lab,range(1,k+1)).max()/a.size))
    return np.array(out)

o_tr, o_te = occl_frac(sample_paths(tr_df,300)), occl_frac(sample_paths(te_df,300))
ths = [0.02,0.05,0.10,0.20]
dfo = pd.DataFrame({"ambang luas":[f">{int(t*100)}%" for t in ths],
                    "TRAIN":[100*(o_tr>t).mean() for t in ths],
                    "TEST":[100*(o_te>t).mean() for t in ths]})
dfo.set_index("ambang luas").plot(kind="bar", figsize=(8,3.6),
                                  color=["steelblue","darkorange"], edgecolor="black")
plt.ylabel("% citra"); plt.title("Citra dgn bercak datar abu-abu (indikasi oklusi buatan)")
plt.xticks(rotation=0); plt.tight_layout(); plt.show()
display(dfo.round(2))

# 18. RINGKASAN JARAK DOMAIN TRAIN -> TEST

In [ ]:
ring = pd.DataFrame([
    dict(ukuran="citra abu-abu murni (%)", TRAIN=100*(s_tr<2).mean(),  TEST=100*(s_te<2).mean()),
    dict(ukuran="ketajaman (median lapvar)", TRAIN=m_tr,               TEST=m_te),
    dict(ukuran="populasi 'blok' AR<3 (%)",  TRAIN=100*(ar_tr<3).mean(),TEST=100*(ar_te<3).mean()),
    dict(ukuran="oklusi >2% luas (%)",       TRAIN=100*(o_tr>0.02).mean(), TEST=100*(o_te>0.02).mean()),
    dict(ukuran="brightness rata-rata",      TRAIN=dfbc.brightness.mean(), TEST=b_te.mean()),
    dict(ukuran="contrast rata-rata",        TRAIN=dfbc.contrast.mean(),   TEST=c_te.mean()),
])
ring["rasio TEST/TRAIN"] = ring.TEST / ring.TRAIN.replace(0, np.nan)
display(ring.round(2))

print("KESIMPULAN")
print("-"*64)
print("1. Warna: train hampir seluruhnya abu-abu, test mayoritas berwarna.")
print("   -> paksa GRAYSCALE pada train DAN test; sumbu warna jadi tak terpakai.")
print("2. Ketajaman: test jauh lebih buram.")
print("   -> augmentasi blur pada train, DITERA ke angka di atas (jangan ditebak).")
print("3. Populasi: porsi 'blok' berbeda -> OOF polos terlalu optimistis.")
print("   -> laporkan macro-F1 terpisah strip/blok, timbang ke komposisi TEST.")
print("4. Oklusi buatan hanya ada di test.")
print("5. Orientasi bermakna (lihat bagian 10) -> JANGAN pakai flip sbg augmentasi.")